# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [59]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [60]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [61]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [62]:
# TODO: Load environment variables
load_dotenv()

True

### VectorDB Instance

In [63]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
chroma_client = chromadb.PersistentClient(path="/workspace/code/project/starter/chromadb")

### Collection

In [64]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base="https://openai.vocareum.com/v1"
)

In [65]:
# TODO: Create a collection
COLLECTION_NAME = "udaplay"

collection = chroma_client.get_or_create_collection(COLLECTION_NAME, embedding_function=embedding_fn)
   

### Add documents

In [66]:
# Make sure you have a directory "project/starter/games"
def function_add_collection(collection):
    data_dir = "games"

    for file_name in sorted(os.listdir(data_dir)):
        if not file_name.endswith(".json"):
            continue

        file_path = os.path.join(data_dir, file_name)
        with open(file_path, "r", encoding="utf-8") as f:
            game = json.load(f)

        # You can change what text you want to index
            content = f"""Platform: [{game['Platform']}]
        Genre: {game['Genre']}
        Name: {game['Name']}
        Release Date: ({game['YearOfRelease']})
        Publisher: {game['Publisher']}
        Description: {game['Description']}
            """


            # Use file name (like 001) as ID
            doc_id = os.path.splitext(file_name)[0]

            collection.add(
                ids=[doc_id],
                documents=[content],
                metadatas=[game]
                # embeddings=[[0.0] * 10] ,
            )



In [67]:
chroma_client = chromadb.PersistentClient(path="chromadb")

embedding_fn = embedding_functions.DefaultEmbeddingFunction()
try:
    chroma_client.delete_collection("udaplay")
except Exception:
    pass
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn
)

function_add_collection(collection)

print("Total docs:", collection.count())



print(results)
print("8888 all records 8888")

results = collection.query(
    query_texts=["racing game for Nintendo Switch"],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(results)

Total docs: 15
{'ids': [['012', '003', '001']], 'embeddings': None, 'documents': [['Platform: [Nintendo Switch]\n        Genre: Racing\n        Name: Mario Kart 8 Deluxe\n        Release Date: (2017)\n        Publisher: Nintendo\n        Description: An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.\n            ', 'Platform: [PlayStation 3]\n        Genre: Racing\n        Name: Gran Turismo 5\n        Release Date: (2010)\n        Publisher: Sony Computer Entertainment\n        Description: A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.\n            ', 'Platform: [PlayStation 1]\n        Genre: Racing\n        Name: Gran Turismo\n        Release Date: (1997)\n        Publisher: Sony Computer Entertainment\n        Description: A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.\n            ']], 'uris': Non